# Подготовка текстовых данных

В электронную приёмную приходят обращения о библиотеке, общежитии, поступлении и работе учебного портала. Позже мы обучим модель направлять их в нужный отдел. Сначала разберёмся с самими сообщениями: пустые строки, повторы и неверные метки могут повлиять на результат ещё до обучения.

В этой работе вы проверите небольшой набор обращений и подготовите функцию обработки текста. Исходные файлы сохраняйте: исправления будем записывать в новые столбцы.

In [3]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Данные

В `requests.csv` собраны обращения с шестью тематическими метками. `group_id` объединяет разные формулировки одного обращения. В отдельном файле `audit_requests.csv` есть несколько ошибок — их и предстоит найти. Описание полей и список меток лежат в `manifest.json`.

Это учебные данные, составленные для курса. Посмотрите на первые строки и попробуйте заметить подозрительные записи до того, как начнёте писать проверки.

In [4]:
requests = pd.read_csv(DATA / "requests.csv", keep_default_na=False)
audit = pd.read_csv(DATA / "audit_requests.csv", keep_default_na=False)
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
required = {"id", "text", "label", "group_id", "split"}
assert required <= set(requests.columns), "Неверная схема основного корпуса"
assert required <= set(audit.columns), "Неверная схема файла аудита"
print("Основной корпус:", requests.shape)
print("Аудит:", audit.shape)
display(audit.head())

Основной корпус: (600, 5)
Аудит: (24, 5)


,id,text,label,group_id,split
0,it-01-1,После смены пароля не могу войти в личный каби...,it,it-01,train
1,it-01-2,Здравствуйте! После смены пароля не могу войти...,it,it-01,train
2,it-01-3,"Подскажите, пожалуйста. После смены пароля не ...",it,it-01,train
3,library-01-1,Как продлить срок пользования учебником?,library,library-01,train
4,library-01-2,Здравствуйте! Как продлить срок пользования уч...,library,library-01,train


### Сначала небольшой пример

Пустой текст не всегда выглядит как пустая строка. Сравним длину трёх значений до и после `strip()`. Какое из них проверка по исходной длине пропустила бы?

In [5]:
sample = pd.Series(["", "   ", "Ошибка 403"], name="text")
display(pd.DataFrame({"text":sample.map(repr),
                      "length":sample.str.len(),
                      "length_after_strip":sample.str.strip().str.len()}))

,text,length,length_after_strip
0,'',0,0
1,' ',3,0
2,'Ошибка 403',10,10


У строки из трёх пробелов исходная длина равна 3, а после `strip()` — 0. Сообщение с кодом ошибки остаётся непустым. Используйте это различие при проверке полной таблицы.

## 1. Что находится в корпусе

Посчитайте строки и группы обращений в основной таблице. Сколько примеров относится к каждой категории? Сравните эти числа: пять перефразировок одного вопроса дают пять строк, но описывают одну ситуацию.

Теперь проверьте файл аудита. Найдите строки без текста и метки, которых нет в списке `manifest["labels"]`. Строку из одних пробелов также считайте пустой.

Запишите результаты в словарь `summary` с полями `rows`, `groups`, `labels`, `empty_rows` и `unknown_labels`. В `labels` поместите частоты категорий, в `empty_rows` — целое число пустых строк, в `unknown_labels` — список неизвестных меток. Выведите сами подозрительные записи, чтобы было понятно, что нашла проверка.

In [28]:
#Ваш код
rows = len(requests) 

groups = requests["group_id"].nunique()

label_counts = requests["label"].value_counts()
labels = {}
for name in label_counts.index:
    labels[name] = int(label_counts[name])

stripped = audit["text"].str.strip()
is_empty = stripped == ""
empty_rows = int(is_empty.sum())

allowed = list(manifest["labels"].keys())
is_known = audit["label"].isin(allowed)
is_unknown = ~is_known
unknown_labels = list(audit.loc[is_unknown, "label"].unique())

summary = {
    "rows": rows,
    "groups": groups,
    "labels": labels,
    "empty_rows": empty_rows,
    "unknown_labels": unknown_labels,
}

display(summary)
display(audit[is_empty])
display(audit[is_unknown])

{'rows': 600,
 'groups': 120,
 'labels': {'it': 100,
  'library': 100,
  'dormitory': 100,
  'study': 100,
  'admission': 100,
  'scholarship': 100},
 'empty_rows': 2,
 'unknown_labels': ['unknown']}

,id,text,label,group_id,split,normalized
19,audit-empty,,it,audit-b,audit,
20,audit-spaces,\t,study,audit-c,audit,


,id,text,label,group_id,split,normalized
22,audit-unknown,Как войти в кабинет?,unknown,audit-e,audit,Как войти в кабинет?


In [16]:
assert summary["rows"] == len(requests)
assert sum(summary["labels"].values()) == len(requests)
assert isinstance(summary["empty_rows"], int)

## 2. Пробелы и Unicode

Начнём с обработки, которая не меняет содержание обращения. Например, строка `"  Не  отменяйте запись!  "` должна превратиться в `"Не отменяйте запись!"`: лишние пробелы исчезли, отрицание и знак остались.

Напишите `normalize_text(text)`. Функция должна приводить Unicode к NFC, заменять последовательность пробельных символов одним пробелом и убирать пробелы по краям. Если передано не строковое значение, вызовите `TypeError`. Регистр, цифры и пунктуацию сохраняйте.

Примените функцию к `audit["text"]`, сохранив результат в новом столбце `audit["normalized"]`. Сопоставьте исходные и обработанные строки: какие из них изменились?

In [26]:
def normalize_text(text):
    if not isinstance(text, str):
        raise TypeError("Expected string")
    text = unicodedata.normalize("NFC", text)
    words = text.split()
    text = " ".join(words)
    return text

original = audit["text"].copy()
audit["normalized"] = audit["text"].map(normalize_text)
changed = audit["text"] != audit["normalized"]
display(audit[changed])

,id,text,label,group_id,split,normalized
18,audit-duplicate,После смены пароля не могу войти в личный каби...,it,audit-a,audit,После смены пароля не могу войти в личный каби...
20,audit-spaces,\t,study,audit-c,audit,


In [45]:
for value in ["  Не  отменяйте!  ", "Ошибка 403", "и\u0306", "", "Ошибка\t401 \n\n при входе "]:
    result = normalize_text(value)
    display(result)
    assert normalize_text(result) == result  # идемпотентность
assert normalize_text("  Не  отменяйте!  ") == "Не отменяйте!"
assert normalize_text("Ошибка 403") == "Ошибка 403"
assert normalize_text("и\u0306") == "й"
assert audit["text"].equals(original)
try:
    normalize_text(None)
except TypeError:
    pass
else:
    raise AssertionError("Нестрочный вход должен вызывать TypeError")

'Не отменяйте!'

'Ошибка 403'

'й'

''

'Ошибка 401 при входе'

## 3. Повторы и разметка

После нормализации некоторые записи могут оказаться одинаковыми. Найдите такие строки среди непустых текстов и сохраните их в `duplicate_rows`. Нужны все участники каждого повтора, включая первую запись.

Для каждого текста посчитайте число разных меток. Оставьте случаи, где их больше одной, в `conflicts`: индекс — нормализованный текст, значение — число меток.

Прочитайте найденные обращения. Где достаточно убрать копию, а где придётся выяснять правильную категорию? Посмотрите и на остальные записи аудита: совпадение текстов — не единственный способ заметить ошибку разметки.

In [34]:
#Ваш код
is_not_empty = audit["normalized"] != ""
filled = audit[is_not_empty]
is_repeat = filled["normalized"].duplicated(keep=False)

labels = filled.groupby("normalized")["label"].nunique()

duplicate_rows = filled[is_repeat]
conflicts = labels[labels > 1]

display(duplicate_rows)
display(conflicts)

,id,text,label,group_id,split,normalized
0,it-01-1,После смены пароля не могу войти в личный каби...,it,it-01,train,После смены пароля не могу войти в личный каби...
18,audit-duplicate,После смены пароля не могу войти в личный каби...,it,audit-a,audit,После смены пароля не могу войти в личный каби...
23,audit-conflict,После смены пароля не могу войти в личный каби...,library,audit-f,audit,После смены пароля не могу войти в личный каби...


normalized
После смены пароля не могу войти в личный кабинет.    2
Name: label, dtype: int64

In [ ]:
assert not duplicate_rows["normalized"].eq("").any()
assert {"it-01-1", "audit-duplicate", "audit-conflict"} <= set(duplicate_rows["id"])
assert len(conflicts) > 0 and (conflicts > 1).all()

## 4. Где проходит граница токена

Сравним два способа разбить сообщение на части. `split()` делит строку по пробелам. Выражение `re.findall(r"\w+|[^\w\s]", text)` выделяет последовательности букв, цифр и подчёркиваний, а также отдельные знаки.

Для каждого примера создайте строку таблицы `token_report`: исходный `text`, токены `split` и `regex`, их количества `n_split` и `n_regex`, а также число разных токенов `vocab_regex`.

Обратите внимание на `help@sever.example`: остался ли адрес одним элементом? Какие элементы относятся к самому адресу, а какие — к окружающей пунктуации? Что нужно добавить к разбиению, чтобы получить адрес целиком и без запятой?

In [35]:
examples = ["ИТ-поддержка: вход не работает!",
            "Письмо: help@sever.example, ошибка 403",
            "книга книга журнал"]
# Ваш код
rows = []

for text in examples:
    split_tokens = text.split()
    regex_tokens = re.findall(r"\w+|[^\w\s]", text)

    unique_tokens = set(regex_tokens)

    row = {
        "text": text,
        "split": split_tokens,
        "regex": regex_tokens,
        "n_split": len(split_tokens),
        "n_regex": len(regex_tokens),
        "vocab_regex": len(unique_tokens),
    }
    rows.append(row)

token_report = pd.DataFrame(rows)
display(token_report)

,text,split,regex,n_split,n_regex,vocab_regex
0,ИТ-поддержка: вход не работает!,"[ИТ-поддержка:, вход, не, работает!]","[ИТ, -, поддержка, :, вход, не, работает, !]",4,8,8
1,"Письмо: help@sever.example, ошибка 403","[Письмо:, help@sever.example,, ошибка, 403]","[Письмо, :, help, @, sever, ., example, ,, оши...",4,10,10
2,книга книга журнал,"[книга, книга, журнал]","[книга, книга, журнал]",3,3,2


In [36]:
assert len(token_report) == len(examples)
assert all(len(tokens) == n for tokens, n in zip(token_report["regex"], token_report["n_regex"]))

Нужно в начало регулярки добавить проверку на email, что-то вроде `[\w.]+@[\w.]`

## 5. Что теряется при очистке текста

Проверим распространённую идею: оставить только слова, перевести их в нижний регистр и удалить частые служебные элементы.

Напишите `aggressive(text)`: выделите токены через `re.findall(r"\w+", text.lower())`, удалите «я», «не» и токены, состоящие только из цифр. Остальное соедините пробелами.

Сначала предскажите результат для трёх строк ниже, затем выполните код. Можно ли по полученным строкам отличить довольного пользователя от недовольного? Что теперь известно о технической ошибке?

In [39]:
def aggressive(text):
    lowered = text.lower()
    tokens = re.findall(r"\w+", lowered)

    result = []
    for token in tokens:
        if token == "я":
            continue
        if token == "не":
            continue
        if token.isdigit():
            continue
        result.append(token)
    return " ".join(result)

texts = ["Я доволен работой портала", "Я не доволен работой портала", "Ошибка 403"]
for text in texts:
    print(text, "=>", aggressive(text))

Я доволен работой портала => доволен работой портала
Я не доволен работой портала => доволен работой портала
Ошибка 403 => ошибка


In [40]:
assert aggressive(texts[0]) == aggressive(texts[1])
assert "403" not in aggressive(texts[2])

## Одной форме могут соответствовать разные слова

Попробуем связать словоформы с леммами. Для «книги» и «книгой» в нашем маленьком словаре один кандидат. Для «стали» — два. Прочитайте оба предложения под выводом и выберите подходящую лемму для каждого. Какие соседние слова помогли сделать выбор?

In [41]:
# Ограниченный учебный словарь: демонстрирует неоднозначность, не заменяет анализатор.
lemma_candidates = {"книги": ["книга"], "книгой": ["книга"], "стали": ["сталь", "стать"]}
for token, candidates in lemma_candidates.items():
    print(token, "=>", candidates)
print("Разберите по контексту: Образцы стали / Студенты стали проверять")

книги => ['книга']
книгой => ['книга']
стали => ['сталь', 'стать']
Разберите по контексту: Образцы стали / Студенты стали проверять


## Что получилось

Кратко опишите найденные проблемы. Для каждой приведите запись или пару записей, объясните, что с ними не так, и предложите решение. Отдельно ответьте на вопросы:

- Почему в корпусе стоит считать и строки, и группы обращений?
- Какое различие потерялось после удаления «не»? Что произошло с кодом ошибки?
- Почему для слова «стали» недостаточно списка возможных лемм?

Добавьте свой пример, на котором функцию нормализации стоило бы проверить. Затем перезапустите ядро и выполните ноутбук целиком. На сдачу нужен этот ноутбук с результатами и вашими пояснениями; отдельный отчёт по тем же пунктам не требуется.

В файле аудита нашлись пустые тексты(audit-spaces), проверка по исходной длине её пропускает, поэтому пустоту нужно проверять после strip(). У записи audit-unknown метка unknown не входит в manifest["labels"], её нужно исправить или исключить. После нормализации совпали тексты it-01-1 и audit-duplicate, можно дубликат удалить так как у них одна метка, а у записи audit-conflict текст тот же, но другая метка, нужно в ручную менять по хорошему. Запись audit-label имеет неправильную метку, её нужно перепроверить вручную.

Ответы:
* Чтобы более точную картину получить, хоть строк сильно больше, по факту они описыают в 5 раз меньше групп обращений
* Не возможно определить отрицание, и настроение сообщения. Число в теории могло позволить определить проблему конкретнее, например 500 ошибка  и 403 оюзначают разное, можно уже классифицировать
* Потому что для этого и некоторых других слов, для определения значения необходимо смотреть на пердложение и соседние слова, т.е. на контекст